# FLUJOS

Construcción del flujo de caja libre para la firma del año base

$$FCFF = EBIT(1-t) - CapEx + D\&A - \Delta CT$$

El NOPAT ya se sabe o se puede obtener por el año base en traxion_anual.csv y de bitacora.md, vease:

**AÑO BASE**

Ingresos base: 38082.2

Margen operativo supuesto: 6.45%

EBIT base: 38,082.2 * 6.45% = 2,456.3


De esta forma, 

$$NOPAT = EBIT \times (1 - t_{\text{marginal}}) = 2{,}456.3 \times (1 - 30\%) = 1{,}719.4$$

La tasa del 30% es la marginal de Mexico proveniente de countrytaxrates.xls

Se necesita la reinversión para poder construir todo el FCFF, proposito de este notebook.


In [5]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
from src.datos import cargar_serie

ebit_base = 2456.3
tasa_mx = 0.30
wacc = 0.1234

nopat = ebit_base * (1 - tasa_mx)
nopat

1719.41

In [6]:
serie = cargar_serie("../data/interim/traxion_anual.csv")

serie["capex_total"] = serie["capex_flota"] + serie["capex_adquisiciones"].fillna(0)
serie["capex_neto"] = serie["capex_total"] - serie["dep_amort"]
serie["capex_neto_sin_adq"] = serie["capex_flota"] - serie["dep_amort"]

serie[["anio", "capex_flota", "capex_adquisiciones", "dep_amort",
       "capex_neto", "capex_neto_sin_adq"]].round(1)

,anio,capex_flota,capex_adquisiciones,dep_amort,capex_neto,capex_neto_sin_adq
0,2021.0,1934.7,0.0,1503.1,431.6,431.6
1,2022.0,3390.1,1633.5,1914.5,3109.1,1475.6
2,2023.0,3434.0,61.3,2238.9,1256.4,1195.1
3,2024.0,3411.9,36.6,2512.0,936.5,899.9
4,2025.0,2348.5,1541.2,2869.4,1020.3,-520.9
5,2026.5,1881.1,1541.2,3111.4,310.9,-1230.3


DYA es estable al rededor de 8 o 9% de ingresos. El capex si cae bastante. El capex neto sin adquisiciones pasó de 1,475.6 en 2022 a -1,230.3 en los UDM.

**CAPEX**

In [7]:
#productividad por segmento
seg = {
    "logistica": {"ing26": 9073, "ing25": 4622, "flota26": 281,  "flota25": 320},
    "carga":     {"ing26": 3679, "ing25": 4107, "flota26": 2258, "flota25": 2272},
    "personas":  {"ing26": 5656, "ing25": 5411, "flota26": 8453, "flota25": 8578},
}

prod = pd.DataFrame(seg).T
prod["ing_x_unidad_25"] = prod["ing25"] / prod["flota25"]
prod["ing_x_unidad_26"] = prod["ing26"] / prod["flota26"]
prod["variacion"] = prod["ing_x_unidad_26"] / prod["ing_x_unidad_25"] - 1

prod[["ing_x_unidad_25", "ing_x_unidad_26", "variacion"]].round(3)

,ing_x_unidad_25,ing_x_unidad_26,variacion
logistica,14.444,32.288,1.235
carga,1.808,1.629,-0.099
personas,0.631,0.669,0.061


El consolidado sube.

Carga cae 9.9% por unidad.

Personas aumenta 6.1% por unidad.

In [11]:
activo_fijo = pd.DataFrame({
    "fecha": ["dic-2023", "dic-2024", "dic-2025", "jun-2026"],
    "equipo_transporte_neto": [14321.8, 15700.9, 16596.0, 16446.2],
    "derecho_de_uso_neto": [1386.3, 1166.3, 2061.6, 1947.4],
})
activo_fijo["total"] = activo_fijo["equipo_transporte_neto"] + activo_fijo["derecho_de_uso_neto"]
activo_fijo["variacion"] = activo_fijo["total"].pct_change()

activo_fijo["variacion"] = (activo_fijo["total"].pct_change() * 100).round(1)
activo_fijo.round(1)

#equipo de transporte neto de traxion_2025_anual.pdf ya que la tabla comparativa trae los tres cierres anuales; derecho de uso de la nota de arrendamientos, 
# donde el saldo al 1 de enero de 2024 sirve como cierre de 2023. Junio 2026 del 2T26.

,fecha,equipo_transporte_neto,derecho_de_uso_neto,total,variacion
0,dic-2023,14321.8,1386.3,15708.1,NaN
1,dic-2024,15700.9,1166.3,16867.2,7.4
2,dic-2025,16596.0,2061.6,18657.6,10.6
3,jun-2026,16446.2,1947.4,18393.6,-1.4


a base de activos fijos creció 7.4% en 2024 y 10.6% en 2025, y solo cayo 1.4% en el primer semestre de 2026. No hay evidencia de que la empresa esté consumiendo su capacidad instalada.

El salto del activo por derecho de uso entre 2024 y 2025, de 1,166.3 a
2,061.6, se debe a Solistica: la nota de
arrendamientos registra 1,094.2 de adiciones por adquisición de negocios.
Traxión no compró esos almacenes, los heredó arrendados.

el capex del estado de flujos parece bajo frente a la D&A total por esto.

**CAPITAL DE TRABAJO**

$$CT_{\text{no monetario}} = \left(CxC + \text{Inventarios} + \text{Otros activos op.}\right) - \left(\text{Proveedores} + \text{Acreedores} + \text{Impuestos por pagar} + \text{Otros pasivos op}\right)$$



In [12]:
balance = pd.DataFrame({
    "fecha": ["dic-2025", "jun-2026"],
    "cuentas_por_cobrar": [6874.131, 7329.079],
    "otras_cuentas_por_cobrar": [443.209, 478.514],
    "inventarios": [295.217, 370.044],
    "pagos_anticipados": [593.949, 754.634],
    "activos_impuestos": [255.336, 259.191],
    "otros_activos_impuestos": [560.914, 545.477],
    "proveedores": [3059.505, 3058.856],
    "acreedores": [1023.743, 966.969],
    "otros_impuestos_por_pagar": [1250.312, 1296.882],
    "pasivos_acumulados": [1608.679, 2269.195],
    "impuesto_utilidad": [108.568, 112.373],
    "ptu": [123.891, 76.499],
    "anticipos_clientes": [66.340, 12.950],
})

activos = ["cuentas_por_cobrar", "otras_cuentas_por_cobrar", "inventarios",
           "pagos_anticipados", "activos_impuestos", "otros_activos_impuestos"]
pasivos = ["proveedores", "acreedores", "otros_impuestos_por_pagar",
           "pasivos_acumulados", "impuesto_utilidad", "ptu", "anticipos_clientes"]

balance["activos_op"] = balance[activos].sum(axis=1)
balance["pasivos_op"] = balance[pasivos].sum(axis=1)
balance["ctno"] = balance["activos_op"] - balance["pasivos_op"]

balance[["fecha", "activos_op", "pasivos_op", "ctno"]].round(1)

,fecha,activos_op,pasivos_op,ctno
0,dic-2025,9022.8,7241.0,1781.7
1,jun-2026,9736.9,7793.7,1943.2
